In [ ]:
import pandas as pd
import joblib
import json
from datetime import datetime
from pathlib import Path
from ollama import chat
from ollamabs.retriever import find_info

        Accuracy  Precision     Recall   F1-score        AUC  n_estimators 
count  30.000000  30.000000  30.000000  30.000000  30.000000     30.000000   
mean    0.873333   0.886248   0.873333   0.871877   0.976703    346.666667   
std     0.038777   0.036129   0.038777   0.039840   0.008340    145.586408   
min     0.800000   0.821869   0.800000   0.796626   0.959171    100.000000   
25%     0.847222   0.861336   0.847222   0.844200   0.970577    200.000000   
50%     0.883333   0.898180   0.883333   0.882557   0.976285    300.000000   
75%     0.897222   0.907846   0.897222   0.896751   0.982747    500.000000   
max     0.966667   0.970370   0.966667   0.966580   0.991655    500.000000   

In [21]:
# Load saved model
model_bundle = joblib.load(
    "final_baseline_model/asl_random_forest.joblib"
)

model = model_bundle['model']
feature_columns = model_bundle['feature_columns']

print("Model loaded successfully!")

Model loaded successfully!


In [70]:
unseen_data = pd.read_csv('unseen_data.csv')
unseen_data.iloc[:, 1:]

,label,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,f510,f511,f512,f513,f514,f515,f516,f517,f518,f519
0,orientation,0.0,0.0,0.0,0.281104,-0.097557,-0.034506,0.511712,-0.265711,-0.090965,...,0.092968,2.378334,0.043664,0.131807,2.814897,0.070089,0.585849,0.923768,0.572137,0.981492
1,key,0.0,0.0,0.0,0.310891,-0.217590,0.022592,0.648923,-0.301535,-0.015425,...,0.879464,0.819811,-0.311274,1.008395,0.833757,-0.278067,0.462589,0.955774,0.631749,0.832801
2,meal,0.0,0.0,0.0,0.244253,-0.218960,-0.127397,0.549937,-0.444205,-0.268251,...,5.887197,16.447224,-0.512717,4.958966,13.717524,-0.411406,0.716158,0.900114,0.783877,0.825911
3,plan,0.0,0.0,0.0,0.052688,-0.592532,-0.143853,0.135745,-1.252859,-0.232586,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.672723,0.893844,0.000000,0.000000
4,long,0.0,0.0,0.0,-0.249392,-0.831487,-0.075865,-0.674760,-1.308590,-0.116643,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.688617,0.941983,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,how,0.0,0.0,0.0,-0.055881,-2.439197,-0.826541,-0.181886,-4.925410,-1.477087,...,-0.105020,-0.492639,-0.140936,-0.040678,-0.435441,-0.115665,0.521196,0.960791,0.643060,1.034626
86,meal,0.0,0.0,0.0,-0.644863,0.007571,-0.311111,-1.195996,-0.070239,-0.622739,...,0.062886,7.485130,-0.166913,0.141750,8.340493,-0.094813,0.736372,0.895503,0.331176,0.816353
87,order,0.0,0.0,0.0,0.172571,-0.229187,-0.058848,0.267662,-0.550977,-0.085650,...,-0.026015,0.771480,-0.123428,-0.039140,0.866663,-0.111024,0.651220,0.894438,0.645494,0.862984
88,meal,0.0,0.0,0.0,-0.176022,-0.362508,-0.050849,-0.280208,-0.754321,-0.128339,...,0.664474,0.590669,-0.263194,0.635981,0.531224,-0.204931,0.839749,1.048873,0.236780,1.057108


In [25]:
input_1 = unseen_data[unseen_data['label'] == 'how'].head(1)
input_2 = unseen_data[unseen_data['label'] == 'order'].head(2)
input_3 = unseen_data[unseen_data['label'] == 'book'].head(1)

In [28]:
sentence, data = ([input_1['label'].values[0], input_2['label'].values[0], input_3['label'].values[0]], 
                  [input_1.iloc[:, 2:], input_2.iloc[1:, 2:], input_3.iloc[:, 2:]])

In [29]:
for word, row in zip(sentence, data):
    # Predict signs
    predictions = model.predict(row)

    # Get prediction probabilities
    probabilities = model.predict_proba(row)

    # Display results
    for prediction, probability in zip(
        predictions, probabilities
    ):
        confidence = probability.max()

        print(
            f"Real sign: {word} | "
            f"Predicted sign: {prediction} | "
            f"Confidence: {confidence:.2%}"
        )


Real sign: how | Predicted sign: how | Confidence: 81.51%
Real sign: order | Predicted sign: order | Confidence: 80.08%
Real sign: book | Predicted sign: book | Confidence: 79.95%


In [49]:
def predict_sign(row): 
    # Predict sign
    prediction = model.predict(row)
    confidence = model.predict_proba(row).max()


    return (prediction[0], float(confidence))

In [50]:
predictions = []
for row in data:
    predictions.append(predict_sign(row))

predictions

[('how', 0.8150555555555556),
 ('order', 0.8007777777777779),
 ('book', 0.799484126984127)]

In [ ]:
MIN_CONFIDENCE = 0.70

PROMPTS_DIR = Path("ollamabs/prompts")
INSTRUCTIONS = (PROMPTS_DIR / "instructions.txt").read_text(encoding="utf-8")
EXAMPLES = json.loads((PROMPTS_DIR / "examples.json").read_text(encoding="utf-8"))


def respond(signs):
    if any(conf < MIN_CONFIDENCE for _, conf in signs):
        return "Sorry, I didn't catch that. Please sign again."

    words = [word for word, _ in signs]
    info = find_info(words) or "No information available for this question."
    now = datetime.now().strftime("%A, %B %d, %Y, %I:%M %p")

    prompt = f"Current time: {now}\n\nCampus info:\n{info}\n\nStudent signed: {' '.join(words)}"

    reply = chat(
        model="gemma4:e2b",
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            *EXAMPLES,
            {"role": "user", "content": prompt},
        ],
        think=False,
        options={"temperature": 0.2, "num_predict": 150},
    )
    return reply["message"]["content"]





In [69]:
print(respond(predictions), "\n")

You can order books at the UTampa Bookstore in the Vaughn Center. 

